# Sponsored Search & User Attention — Code Implementations
### Based on: *Sponsored Search and User Attention: A Conceptual Exploration of Influence and Trust*
Chitkara University Institute of Engineering and Technology

---

This notebook contains two executable code sections tied to the research paper:

1. **AUC / Classifier Simulation** — Replicates the Random Forest relevance model from Aiello et al. (2016)
2. **CTR vs. Relevance Analysis** — Visualises the mismatch between click-through rate and ad relevance

> **Requirements:** `scikit-learn`, `pandas`, `numpy`, `matplotlib`, `seaborn`  
> Install with: `pip install scikit-learn pandas numpy matplotlib seaborn`


---
## Section 1 — AUC / Classifier Simulation
### (Based on Section 2.2 of the paper — Relevance vs. Clickability)

**Background:**  
Aiello et al. (2016) trained a Random Forest classifier on 185 textual features to predict ad relevance,
*independent of click history*. It achieved:
- Random Forest (185 features): **AUC = 0.764**
- Basic text baseline (19 features): **AUC = 0.667**
- Click-augmented baseline: **AUC = 0.595**

This section simulates that experiment using a synthetic dataset that mirrors the study's structure.

**Dataset:** Synthetically generated using `sklearn.datasets.make_classification`.  
Each sample represents one (query, ad) pair. Features represent textual similarity signals
(e.g. cosine similarity, BM25 score, Jaccard coefficient — as described in Aiello et al.).  
Label `1` = relevant ad, `0` = irrelevant ad.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (roc_auc_score, roc_curve, classification_report,
                              ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

print("All libraries imported successfully ✓")


In [ ]:
# ── Dataset Generation ────────────────────────────────────────────────────────
# Simulates a dataset of (query, ad) pairs with textual relevance features.
# Mirrors Aiello et al. (2016): 185 features, binary relevance label.

np.random.seed(42)

N_SAMPLES   = 2000   # number of (query, ad) pairs
N_FEATURES  = 185    # total textual features (as in the paper)
N_BASIC     = 19     # basic text baseline feature count
N_CLICK_AUG = 30     # click-augmented baseline feature count

X_full, y = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=60,
    n_redundant=40,
    n_clusters_per_class=2,
    weights=[0.55, 0.45],   # slight class imbalance (more irrelevant than relevant)
    flip_y=0.05,            # 5% label noise — realistic for editorial judgements
    random_state=42
)

# Sub-feature sets for the two baselines
X_basic     = X_full[:, :N_BASIC]
X_click_aug = X_full[:, :N_CLICK_AUG]

# Show dataset summary
df_summary = pd.DataFrame({
    "Split":    ["Full dataset", "Train (80%)", "Test (20%)"],
    "Samples":  [N_SAMPLES, int(N_SAMPLES * 0.8), int(N_SAMPLES * 0.2)],
    "Relevant (label=1)": [
        y.sum(),
        int(y[:int(N_SAMPLES*0.8)].sum()),
        int(y[int(N_SAMPLES*0.8):].sum())
    ],
    "Irrelevant (label=0)": [
        (y == 0).sum(),
        int((y[:int(N_SAMPLES*0.8)] == 0).sum()),
        int((y[int(N_SAMPLES*0.8):] == 0).sum())
    ]
})
print("Dataset Summary")
print("=" * 50)
print(df_summary.to_string(index=False))
print(f"\nFeature counts → Full: {N_FEATURES} | Basic: {N_BASIC} | Click-aug: {N_CLICK_AUG}")


In [ ]:
# ── Train & Evaluate Three Models ────────────────────────────────────────────

models = {
    "Random Forest\n(185 features)": (
        RandomForestClassifier(n_estimators=200, max_depth=12,
                               min_samples_leaf=5, random_state=42, n_jobs=-1),
        X_full
    ),
    "Basic Text Baseline\n(19 features)": (
        Pipeline([("scaler", StandardScaler()),
                  ("clf", LogisticRegression(C=0.5, max_iter=1000, random_state=42))]),
        X_basic
    ),
    "Click-Augmented Baseline\n(30 features)": (
        Pipeline([("scaler", StandardScaler()),
                  ("clf", LogisticRegression(C=0.5, max_iter=1000, random_state=42))]),
        X_click_aug
    ),
}

results = {}

for name, (model, X) in models.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    model.fit(X_train, y_train)
    y_prob  = model.predict_proba(X_test)[:, 1]
    y_pred  = model.predict(X_test)
    auc     = roc_auc_score(y_test, y_prob)
    fpr, tpr, _ = roc_curve(y_test, y_prob)

    results[name] = {
        "model": model, "X_test": X_test, "y_test": y_test,
        "y_prob": y_prob, "y_pred": y_pred,
        "auc": auc, "fpr": fpr, "tpr": tpr
    }
    print(f"{name.replace(chr(10), ' '):<45}  AUC = {auc:.3f}")

# Paper's reported values for reference
print("\n── Paper reported values (Aiello et al., 2016) ──")
paper_aucs = {"Random Forest (185 features)": 0.764,
              "Basic Text Baseline (19 features)": 0.667,
              "Click-Augmented Baseline (30 features)": 0.595}
for k, v in paper_aucs.items():
    print(f"  {k:<45}  AUC = {v:.3f}")


In [ ]:
# ── ROC Curve Comparison ──────────────────────────────────────────────────────

colors = ["#2ecc71", "#3498db", "#e74c3c"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Simulated AUC curves
ax = axes[0]
for (name, res), color in zip(results.items(), colors):
    label = f"{name.replace(chr(10), ' ')}  (AUC={res['auc']:.3f})"
    ax.plot(res["fpr"], res["tpr"], lw=2, color=color, label=label)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random classifier (AUC=0.500)")
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title("ROC Curves — Simulated (This Notebook)", fontsize=12, fontweight="bold")
ax.legend(fontsize=8, loc="lower right")
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.grid(alpha=0.3)

# Right: Paper vs Simulation bar comparison
ax2 = axes[1]
model_labels = ["Random Forest\n(185 features)", "Basic Text\nBaseline (19)",
                "Click-Augmented\nBaseline (30)"]
paper_vals = [0.764, 0.667, 0.595]
sim_vals   = [results[k]["auc"] for k in results]

x = np.arange(len(model_labels))
w = 0.35
bars1 = ax2.bar(x - w/2, paper_vals, w, label="Paper (Aiello et al., 2016)",
                color="#2c3e50", alpha=0.85, edgecolor="white")
bars2 = ax2.bar(x + w/2, sim_vals,   w, label="This Simulation",
                color="#3498db", alpha=0.85, edgecolor="white")

for bar in bars1:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8, color="#2c3e50")
for bar in bars2:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8, color="#2980b9")

ax2.set_xticks(x); ax2.set_xticklabels(model_labels, fontsize=9)
ax2.set_ylabel("AUC Score", fontsize=11)
ax2.set_title("Paper vs Simulation AUC Comparison", fontsize=12, fontweight="bold")
ax2.legend(fontsize=9); ax2.set_ylim([0.4, 0.9]); ax2.grid(axis="y", alpha=0.3)

plt.suptitle("AUC Results: Relevance Classification of Sponsored Ads",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("auc_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nKey takeaway: The Random Forest on rich textual features consistently")
print("outperforms click-history-based approaches — supporting the paper's argument")
print("that clickability ≠ relevance.")


In [ ]:
# ── Feature Importance (Random Forest) ───────────────────────────────────────
# Shows which textual feature groups drive relevance prediction.
# Feature groups mirror the categories described in Aiello et al. (2016):
# cosine similarity, BM25, Jaccard, LSI, hash embeddings.

rf_model = results["Random Forest\n(185 features)"]["model"]
importances = rf_model.feature_importances_

# Assign feature group labels
group_sizes = {"Cosine Similarity": 30, "BM25 Score": 35, "Jaccard Coefficient": 25,
               "LSI Representation": 50, "Hash Embedding": 45}
group_labels = []
for grp, size in group_sizes.items():
    group_labels.extend([grp] * size)

imp_df = pd.DataFrame({"feature_idx": range(N_FEATURES),
                        "importance": importances,
                        "group": group_labels})
group_imp = imp_df.groupby("group")["importance"].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
palette = ["#2ecc71", "#3498db", "#9b59b6", "#e67e22", "#e74c3c"]
bars = ax.barh(group_imp.index, group_imp.values,
               color=palette, edgecolor="white", height=0.55)
for bar, val in zip(bars, group_imp.values):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=9)
ax.set_xlabel("Total Feature Importance", fontsize=11)
ax.set_title("Feature Group Importance — Random Forest Relevance Classifier\n"
             "(Feature groups based on Aiello et al., 2016)", fontsize=11, fontweight="bold")
ax.set_xlim([0, group_imp.max() + 0.04])
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()


---
## Section 2 — CTR vs. Relevance Analysis
### (Based on Section 2.2 of the paper — Relevance vs. Clickability)

**Background:**  
Aiello et al. (2016) found a striking mismatch between click-through rate (CTR) and actual relevance:

| Finding | Statistic |
|---|---|
| Highly relevant ads with **zero clicks** (CTR = 0) | **60%** |
| High-CTR ads rated as **irrelevant** | **17%** |
| Relevant ads that were also clicked | ~23% |

This reveals that **click data alone is a poor proxy for relevance** — the core argument of Section 2.2.

**Dataset:** Synthetically generated to reflect the above proportions.  
Each row = one sponsored ad. Columns: `relevance_label`, `ctr`, `impressions`, `bid_amount`, `erpm`.


In [ ]:
# ── Build Synthetic Ad Dataset ────────────────────────────────────────────────
# Reproduces the relevance/CTR distribution from Aiello et al. (2016)

np.random.seed(99)
N_ADS = 5000

# ── Assign relevance labels ──
# "Highly Relevant" (HR) = 83%, "Irrelevant" (IR) = 17%
#  (within HR: 60% have CTR=0, 23% have clicks;  all IR ads have some clicks)
relevance = np.random.choice(["Highly Relevant", "Irrelevant"],
                              p=[0.83, 0.17], size=N_ADS)

ctrs, impressions_list, bids = [], [], []

for rel in relevance:
    imp = np.random.randint(100, 50_000)
    bid = round(np.random.uniform(0.5, 5.0), 2)

    if rel == "Highly Relevant":
        # 60% of relevant ads have CTR = 0 (never displayed due to no click history)
        if np.random.rand() < 0.60:
            ctr = 0.0
        else:
            ctr = round(np.random.beta(2, 15), 4)   # low-to-mid CTR
    else:  # Irrelevant
        # 17% of high-CTR ads — irrelevant but well-known brands with rich click history
        ctr = round(np.random.beta(5, 8) * 0.25, 4)   # higher CTR range

    ctrs.append(ctr)
    impressions_list.append(imp)
    bids.append(bid)

df_ads = pd.DataFrame({
    "relevance_label": relevance,
    "ctr":             ctrs,
    "impressions":     impressions_list,
    "bid_amount":      bids,
})
df_ads["clicks"]     = (df_ads["ctr"] * df_ads["impressions"]).astype(int)
df_ads["erpm"]       = (df_ads["ctr"] * df_ads["bid_amount"] * 1000).round(2)
df_ads["ctr_bucket"] = pd.cut(df_ads["ctr"],
                               bins=[-0.001, 0.0, 0.05, 0.15, 1.0],
                               labels=["Zero (CTR=0)", "Low (0–5%)",
                                       "Medium (5–15%)", "High (>15%)"])

print("Dataset shape:", df_ads.shape)
print("\nRelevance distribution:")
print(df_ads["relevance_label"].value_counts())
print("\nSample rows:")
df_ads.head(8)


In [ ]:
# ── Core Mismatch Bar Chart ───────────────────────────────────────────────────
# Reproduces the headline finding: 60% of relevant ads have zero clicks

# Calculate actual percentages from the dataset
hr_ads = df_ads[df_ads["relevance_label"] == "Highly Relevant"]
ir_ads = df_ads[df_ads["relevance_label"] == "Irrelevant"]

hr_zero_pct  = (hr_ads["ctr"] == 0).mean() * 100
ir_hictr_pct = (ir_ads["ctr"] > 0.05).mean() * 100

# Four key categories (mirroring the paper)
categories = [
    "Highly Relevant\nads with CTR = 0",
    "Irrelevant ads\nwith High CTR",
    "Relevant ads\nthat got clicks",
    "Irrelevant ads\nwith Low/Zero CTR"
]
paper_vals = [60, 17, 23, 0]
sim_vals   = [
    round(hr_zero_pct, 1),
    round(ir_hictr_pct, 1),
    round(100 - hr_zero_pct, 1),
    round(100 - ir_hictr_pct, 1)
]
bar_colors = ["#e74c3c", "#e67e22", "#2ecc71", "#95a5a6"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Paper values
ax1 = axes[0]
bars = ax1.bar(categories, paper_vals, color=bar_colors, edgecolor="white",
               width=0.55, linewidth=1.2)
for bar, val in zip(bars, paper_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{val}%", ha="center", fontsize=11, fontweight="bold")
ax1.set_title("Paper Findings\n(Aiello et al., 2016)", fontsize=12, fontweight="bold")
ax1.set_ylabel("Percentage of Ads (%)", fontsize=11)
ax1.set_ylim([0, 78]); ax1.grid(axis="y", alpha=0.3)
ax1.tick_params(axis="x", labelsize=8.5)

# Right: Simulated values
ax2 = axes[1]
bars2 = ax2.bar(categories, sim_vals, color=bar_colors, edgecolor="white",
                width=0.55, linewidth=1.2)
for bar, val in zip(bars2, sim_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{val}%", ha="center", fontsize=11, fontweight="bold")
ax2.set_title("Simulated Dataset\n(This Notebook)", fontsize=12, fontweight="bold")
ax2.set_ylabel("Percentage of Ads (%)", fontsize=11)
ax2.set_ylim([0, 78]); ax2.grid(axis="y", alpha=0.3)
ax2.tick_params(axis="x", labelsize=8.5)

legend_patches = [
    mpatches.Patch(color="#e74c3c", label="Relevant but invisible (CTR=0)"),
    mpatches.Patch(color="#e67e22", label="Irrelevant but clicked"),
    mpatches.Patch(color="#2ecc71", label="Relevant & clicked"),
    mpatches.Patch(color="#95a5a6", label="Irrelevant & not clicked"),
]
fig.legend(handles=legend_patches, loc="lower center", ncol=4,
           fontsize=9, bbox_to_anchor=(0.5, -0.08), frameon=True)

plt.suptitle("CTR vs. Relevance Mismatch in Sponsored Search Ads",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("ctr_relevance_mismatch.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nKey finding reproduced: ~60% of relevant ads never get shown due to zero click history.")
print("~17% of ads with decent CTR are actually irrelevant to the user's query.")


In [ ]:
# ── CTR Distribution by Relevance Group ──────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: CTR bucket breakdown (stacked bar)
ax1 = axes[0]
ct = df_ads.groupby(["relevance_label", "ctr_bucket"]).size().unstack(fill_value=0)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
bucket_colors = ["#e74c3c", "#3498db", "#f39c12", "#2ecc71"]
ct_pct.plot(kind="bar", stacked=True, ax=ax1,
            color=bucket_colors, edgecolor="white", width=0.5)
ax1.set_title("CTR Bucket Distribution by Ad Relevance", fontsize=11, fontweight="bold")
ax1.set_ylabel("Percentage of Ads (%)", fontsize=10)
ax1.set_xlabel("Relevance Label", fontsize=10)
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0, fontsize=10)
ax1.legend(title="CTR Bucket", fontsize=8, bbox_to_anchor=(1.02, 1))
ax1.grid(axis="y", alpha=0.3)

# Right: eRPM distribution
ax2 = axes[1]
for label, color in [("Highly Relevant", "#2ecc71"), ("Irrelevant", "#e74c3c")]:
    subset = df_ads[df_ads["relevance_label"] == label]["erpm"]
    subset_nonzero = subset[subset > 0]
    ax2.hist(subset_nonzero, bins=40, alpha=0.65, color=color,
             label=f"{label} (n={len(subset_nonzero):,})", edgecolor="white")

ax2.set_title("eRPM Distribution: Relevant vs Irrelevant Ads\n"
              "(ads with eRPM > 0 only)", fontsize=11, fontweight="bold")
ax2.set_xlabel("eRPM (expected Revenue per Mille Impressions)", fontsize=10)
ax2.set_ylabel("Number of Ads", fontsize=10)
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("ctr_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nObservation: Irrelevant ads often achieve higher eRPM than relevant ones")
print("because eRPM = CTR × bid — and irrelevant ads from big brands have higher CTR.")
print("This is the core bias described in Section 2.1 of the paper.")


In [ ]:
# ── eRPM Ranking Bias Illustration ───────────────────────────────────────────
# If we rank by eRPM, many relevant ads with CTR=0 are pushed out entirely.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter — CTR vs bid coloured by relevance
sample = df_ads.sample(600, random_state=1)
ax1 = axes[0]
for label, color, marker in [("Highly Relevant", "#2ecc71", "o"),
                               ("Irrelevant", "#e74c3c", "^")]:
    s = sample[sample["relevance_label"] == label]
    ax1.scatter(s["bid_amount"], s["ctr"], alpha=0.45, s=20,
                color=color, marker=marker, label=label)
ax1.set_xlabel("Bid Amount ($)", fontsize=10)
ax1.set_ylabel("Click-Through Rate (CTR)", fontsize=10)
ax1.set_title("Bid Amount vs CTR\n(coloured by relevance label)", fontsize=11, fontweight="bold")
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

# Right: Top-100 by eRPM vs Top-100 by relevance overlap
top100_erpm = df_ads.nlargest(100, "erpm")
top100_rel  = df_ads[df_ads["relevance_label"] == "Highly Relevant"].nlargest(100, "impressions")

erpm_rel_count = (top100_erpm["relevance_label"] == "Highly Relevant").sum()
rel_erpm_count = top100_rel.index.isin(top100_erpm.index).sum()

categories_bar = ["Top 100 by eRPM\n(how many are relevant?)",
                  "Top 100 Relevant ads\n(how many are in eRPM top 100?)"]
values_bar = [erpm_rel_count, rel_erpm_count]
bar_colors2 = ["#e67e22", "#3498db"]

ax2 = axes[1]
bars = ax2.bar(categories_bar, values_bar, color=bar_colors2,
               edgecolor="white", width=0.45)
for bar, val in zip(bars, values_bar):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{val}/100", ha="center", fontsize=12, fontweight="bold")
ax2.axhline(50, ls="--", color="grey", lw=1, label="50% reference line")
ax2.set_title("Ranking Overlap: eRPM vs Relevance\n"
              "(closer to 100 = better alignment)", fontsize=11, fontweight="bold")
ax2.set_ylabel("Count (out of 100)", fontsize=10)
ax2.set_ylim([0, 115]); ax2.legend(fontsize=9); ax2.grid(axis="y", alpha=0.3)
ax2.tick_params(axis="x", labelsize=8.5)

plt.suptitle("eRPM Ranking Bias — Relevant Ads Disadvantaged by Low Click History",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("erpm_bias.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nOf the top 100 ads ranked by eRPM: {erpm_rel_count} are highly relevant.")
print(f"Of the top 100 relevant ads: only {rel_erpm_count} appear in the eRPM top 100.")
print("\nThis quantifies the incumbency bias described in the paper (Section 2.1).")


In [ ]:
# ── Summary Statistics Table ──────────────────────────────────────────────────

summary = df_ads.groupby("relevance_label").agg(
    Total_Ads    = ("ctr", "count"),
    Zero_CTR_pct = ("ctr", lambda x: f"{(x == 0).mean()*100:.1f}%"),
    Avg_CTR      = ("ctr", lambda x: f"{x.mean():.4f}"),
    Avg_eRPM     = ("erpm", lambda x: f"{x.mean():.2f}"),
    Median_eRPM  = ("erpm", lambda x: f"{x.median():.2f}"),
    Avg_Bid      = ("bid_amount", lambda x: f"${x.mean():.2f}"),
).reset_index()

print("=" * 70)
print("SUMMARY: CTR and Revenue Metrics by Relevance Group")
print("=" * 70)
print(summary.to_string(index=False))
print()
print("Key insight: Highly relevant ads have much higher zero-CTR rates,")
print("meaning the eRPM ranking system systematically suppresses them.")
print()

# Section 1 AUC summary
print("=" * 70)
print("SUMMARY: AUC Scores — Classifier Comparison")
print("=" * 70)
print(f"{'Model':<42} {'Paper AUC':>10} {'Simulated AUC':>14}")
print("-" * 70)
paper_auc_vals = [0.764, 0.667, 0.595]
sim_auc_vals   = [results[k]["auc"] for k in results]
model_names_short = ["Random Forest (185 features)",
                     "Basic Text Baseline (19 features)",
                     "Click-Augmented Baseline (30 features)"]
for name, p_auc, s_auc in zip(model_names_short, paper_auc_vals, sim_auc_vals):
    print(f"  {name:<40} {p_auc:>10.3f} {s_auc:>14.3f}")
print()
print("Conclusion: Rich textual features (independent of click history) produce")
print("better relevance predictions — supporting transparency in sponsored search.")


---
## References

- Aiello, L.M., et al. (2016). *The role of relevance in sponsored search.* CIKM 2016. https://doi.org/10.1145/2983323.2983840  
- Danescu-Niculescu-Mizil, C., et al. (2010). *Competing for users' attention.* WWW 2010. https://doi.org/10.1145/1772690.1772721  
- Lu, Y., et al. (2017). *Are sponsored links effective?* ACM TMIS. https://doi.org/10.1145/3023365  

---
*Notebook prepared as supplementary code for the research paper:  
"Sponsored Search and User Attention: A Conceptual Exploration of Influence and Trust"  
Chitkara University Institute of Engineering and Technology, Punjab, India.*
